In [1]:
import numpy as np

In [2]:
# parameterise a line

def point_on_line(a, b, t):

    point = a + t*(b-a)

    return point

def direction_vector(a,b):
    # Direction from a to b
    return b-a

def nearest_point_line(dist_eval_point, a, b):

    eval_point_direction = direction_vector(a,dist_eval_point)
    line_vector = direction_vector(a,b)

    line_vector_mag = np.linalg.norm(line_vector)

    t_local = np.clip((np.dot(line_vector, eval_point_direction)/(line_vector_mag)**2),0,1)

    line_equation = point_on_line(a,b,t_local)
    distance = np.linalg.norm(dist_eval_point-line_equation)

    return t_local, distance


# Example 

a = np.array([1.0,0.0])

b = np.array([0.0,1.0])

line_vector = direction_vector(a,b)

dist_eval_point = np.array([0.0,0.0])
t_local, distance = nearest_point_line(dist_eval_point, a,b)

print(t_local)

0.4999999999999999


In [3]:
# Finding nearest line and point on the line from set of lines or polyline

def nearest_point_on_polyline(points, eval_point):

    nearest_distance = 1e10
    best_i = 0
    best_t = 0.0

    for i in range(len(points)-1):

        t_local, distance = nearest_point_line(eval_point, points[i], points[i+1])

        if distance < nearest_distance:

            nearest_distance = distance
            best_i = i
            best_t = t_local

    nearest_point = point_on_line(points[best_i], points[best_i+1], best_t)

    

    return best_t, nearest_distance, nearest_point

In [4]:
# L-shaped polyline: three points
points = np.array([
    [0.0, 0.0],
    [2.0, 0.0],
    [2.0, 2.0]
])

# Query point
q = np.array([3.0, 1.0])

t, distance, nearest_point = nearest_point_on_polyline(points, q)

print(nearest_point)

[2. 1.]


Distance from a point to a surface triangle

In [5]:
def triangle_normal_and_area(a, b, c):

    u = b-a
    v = c-a

    area_dir = 0.5*(np.cross(u,v))
    area_normal = area_dir/(np.linalg.norm(area_dir))
    area = np.linalg.norm(area_dir)

    return area_normal, area

a = np.array([1.0, 0.0,0.0])
b = np.array([0.0, 1.0,0.0])
c = np.array([0.0,0.0,0.0])

normal, area = triangle_normal_and_area(a,b,c)

print(normal)
print(area)


def distance_from_triangle(eval_point, plane_point, plane_normal):

    signed_dist = np.dot((eval_point - plane_point), plane_normal)

    point_on_plane = eval_point - signed_dist*plane_normal

    return signed_dist, point_on_plane


eval_point = np.array([0.51,0.499999,6.0])

distance, point_on_plane = distance_from_triangle(eval_point, a, normal)

print(distance)
print(point_on_plane)


def point_in_or_out_triangle(a,b,c,p, normal):

    d1 = np.dot(np.cross((b-a), (p-a)),normal)
    d2 = np.dot(np.cross((c-b), (p-b)),normal)
    d3 = np.dot(np.cross((a-c), (p-c)),normal)

    boolean = (d1<0) or (d2<0) or (d3<0)

    return boolean

answer = point_in_or_out_triangle(a,b,c,point_on_plane, normal)

if answer:

    print(f"Projected point {point_on_plane} is outside the triangle")
else:
    print(f"Projected point {point_on_plane} is inside the triangle")


[0. 0. 1.]
0.5
6.0
[0.51     0.499999 0.      ]
Projected point [0.51     0.499999 0.      ] is outside the triangle


In [ ]:
import jax
import jax.numpy as jnp
from jax import grad,jit
from scipy.optimize import minimize

In [ ]:
from dataclasses import dataclass

@dataclass
class Segment:
    p1: np.ndarray
    p2: np.ndarray


def segment_segment_jax(seg1: Segment, seg2: Segment):

    p1,p2 = jnp.array(seg1.p1), jnp.array(seg1.p2)
    q1, q2 = jnp.array(seg2.p1), jnp.array(seg2.p2)

    def distance(params):

        s,t = jnp.clip(params[0], 0, 1), jnp.clip(params[1], 0, 1)

        P = p1 + s * (p2 - p1)
        Q = q1 + t * (q2 - q1)
        return jnp.sum((P - Q)**2)
    

    grad_fn = jit(grad(distance))

    result = minimize(fun = lambda x: float(distance(jnp.array(x))),
                      )


    


In [ ]:
def gradient_descent_point_to_line(a,b,q,alpha=0.01,steps=1000):

    t = 0.5
    d = b-a
    
    for i in range(steps):
        P=a+t*d
        grad = np.sum((P-q)**2)

        t = t-alpha*grad
        